# 03 Context Window Calculator

This notebook studies how prompt size and reserved output tokens interact with a model context window.

Main formula:

`prompt tokens + reserved output tokens <= context window`

## Step 3 Goal

We are still using only tokenizers in this step. We are not loading full models, generating text, or comparing with the OpenAI API.

We want to understand:
- how many tokens a prompt already uses
- why output-token reservation matters
- which prompts fit or exceed different context windows

In [8]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from context_analyzer import (
    CONTEXT_EXAMPLES_PATH,
    CONTEXT_REPORT_PATH,
    DEFAULT_CONTEXT_WINDOWS,
    DEFAULT_RESERVED_OUTPUT_TOKENS,
    FIGURES_DIR,
    analyze_multiple_contexts,
    build_context_summary,
    create_context_graphs,
    load_context_examples,
    load_all_tokenizers,
    save_context_report,
)
from tokenizer_utils import REQUESTED_MODEL_NAMES

print("Project root:", PROJECT_ROOT)
print("Context examples path:", CONTEXT_EXAMPLES_PATH)
print("Context report path:", CONTEXT_REPORT_PATH)
print("Figures directory:", FIGURES_DIR)

Project root: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference
Context examples path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/examples/context_examples.json
Context report path: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/context_window_report.csv
Figures directory: /Users/nikhiljuluri/Desktop/LLM_Infra_Projects/LLM_Inference_cost_Latency_analyser/week2_LLM_Inference/outputs/figures


## Load Prompt Scenarios

The JSON file stores several realistic prompt shapes, including short prompts, long prompts, code debugging prompts, RAG-style prompts, chat history, JSON, and multilingual text.

In [9]:
prompts = load_context_examples(CONTEXT_EXAMPLES_PATH)
prompts

[{'prompt_name': 'short user question',
  'text': 'What is the difference between tokenization and a context window in large language model inference?'},
 {'prompt_name': 'long essay prompt',
  'text': 'Write a detailed essay explaining how inference works in large language models. Cover tokenization, context windows, prompt structure, prefill, decode, batching, KV cache reuse, latency tradeoffs, and cost awareness. Use simple language, give practical examples, and compare how short prompts and long prompts use the context window differently. Also explain why engineers need to reserve room for output tokens before sending a prompt to a model.'},
 {'prompt_name': 'code debugging prompt',
  'text': 'You are helping debug a Python script. The script loads a tokenizer, counts tokens, and then fails with an index error when processing a long prompt.\n\nTraceback:\nIndexError: list index out of range\nFile "context.py", line 41, in analyze_prompt\n\nPlease explain the likely cause, identify 

## Load Tokenizers

This reuses the tokenizer-loading logic from Step 2. If a requested tokenizer is unavailable, the utility can fall back to a safe backup.

In [10]:
tokenizers = load_all_tokenizers(REQUESTED_MODEL_NAMES)
list(tokenizers.keys())

['Qwen/Qwen2.5-1.5B-Instruct',
 'mistralai/Mistral-7B-Instruct-v0.2',
 'TinyLlama/TinyLlama-1.1B-Chat-v1.0']

## Run the Context Analysis

For each prompt, tokenizer, context window, and reserved output size, we calculate:
- prompt token count
- total required tokens
- remaining tokens
- context used percent
- whether the prompt still fits

In [11]:
results = analyze_multiple_contexts(
    prompts=prompts,
    tokenizers=tokenizers,
    context_windows=DEFAULT_CONTEXT_WINDOWS,
    reserved_output_tokens_list=DEFAULT_RESERVED_OUTPUT_TOKENS,
)
df = save_context_report(results, CONTEXT_REPORT_PATH)
df.head(10)

,prompt_name,tokenizer_name,character_count,word_count,prompt_token_count,reserved_output_tokens,context_window,total_required_tokens,remaining_tokens,context_used_percent,fits_context_window
182,multilingual prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,153,21,163,1024,4096,1187,2909,28.9795,True
101,RAG-style prompt with multiple chunks,TinyLlama/TinyLlama-1.1B-Chat-v1.0,600,89,144,1024,4096,1168,2928,28.5156,True
92,RAG-style prompt with multiple chunks,mistralai/Mistral-7B-Instruct-v0.2,600,89,139,1024,4096,1163,2933,28.3936,True
173,multilingual prompt,mistralai/Mistral-7B-Instruct-v0.2,153,21,124,1024,4096,1148,2948,28.0273,True
83,RAG-style prompt with multiple chunks,Qwen/Qwen2.5-1.5B-Instruct,600,89,119,1024,4096,1143,2953,27.9053,True
164,multilingual prompt,Qwen/Qwen2.5-1.5B-Instruct,153,21,109,1024,4096,1133,2963,27.6611,True
155,JSON/tool-calling prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,357,43,108,1024,4096,1132,2964,27.6367,True
146,JSON/tool-calling prompt,mistralai/Mistral-7B-Instruct-v0.2,357,43,108,1024,4096,1132,2964,27.6367,True
47,long essay prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,447,66,92,1024,4096,1116,2980,27.2461,True
137,JSON/tool-calling prompt,Qwen/Qwen2.5-1.5B-Instruct,357,43,90,1024,4096,1114,2982,27.1973,True


## Full Report Sorted by Context Used Percent

This makes it easy to spot the prompts that stress the context window the most.

In [12]:
df

,prompt_name,tokenizer_name,character_count,word_count,prompt_token_count,reserved_output_tokens,context_window,total_required_tokens,remaining_tokens,context_used_percent,fits_context_window
182,multilingual prompt,TinyLlama/TinyLlama-1.1B-Chat-v1.0,153,21,163,1024,4096,1187,2909,28.9795,True
101,RAG-style prompt with multiple chunks,TinyLlama/TinyLlama-1.1B-Chat-v1.0,600,89,144,1024,4096,1168,2928,28.5156,True
92,RAG-style prompt with multiple chunks,mistralai/Mistral-7B-Instruct-v0.2,600,89,139,1024,4096,1163,2933,28.3936,True
173,multilingual prompt,mistralai/Mistral-7B-Instruct-v0.2,153,21,124,1024,4096,1148,2948,28.0273,True
83,RAG-style prompt with multiple chunks,Qwen/Qwen2.5-1.5B-Instruct,600,89,119,1024,4096,1143,2953,27.9053,True
...,...,...,...,...,...,...,...,...,...,...,...
114,chat history prompt,Qwen/Qwen2.5-1.5B-Instruct,388,59,75,256,32768,331,32437,1.0101,True
60,code debugging prompt,Qwen/Qwen2.5-1.5B-Instruct,322,52,72,256,32768,328,32440,1.0010,True
15,short user question,mistralai/Mistral-7B-Instruct-v0.2,99,15,18,256,32768,274,32494,0.8362,True
6,short user question,Qwen/Qwen2.5-1.5B-Instruct,99,15,17,256,32768,273,32495,0.8331,True


## Summary Findings

These summary values answer the main Step 3 questions:
- which prompt uses the most context
- which tokenizer creates the highest average context usage
- which prompts are tight inside a 4096-token window
- which prompts comfortably fit in 8192
- which prompts need larger windows

In [13]:
summary = build_context_summary(df)
summary

{'most_context_heavy_prompt': 'multilingual prompt',
 'highest_average_context_tokenizer': 'TinyLlama/TinyLlama-1.1B-Chat-v1.0',
 'lowest_average_context_tokenizer': 'Qwen/Qwen2.5-1.5B-Instruct',
 'tight_4096_prompts': [],
 'prompts_that_fit_8192': ['multilingual prompt',
  'RAG-style prompt with multiple chunks',
  'JSON/tool-calling prompt',
  'long essay prompt',
  'chat history prompt',
  'code debugging prompt',
  'short user question'],
 'prompts_that_need_larger_context_windows': []}

## Create Graphs

The graph helper uses the `4096` context window and `512` reserved output tokens as the default comparison slice so the charts stay easy to read.

In [14]:
create_context_graphs(df, FIGURES_DIR)
sorted(path.name for path in FIGURES_DIR.glob('*.png'))

['character_count_vs_token_count.png',
 'context_usage_by_prompt.png',
 'context_used_percent_by_tokenizer.png',
 'fits_vs_exceeds_context.png',
 'included_vs_dropped_sections.png',
 'packed_prompt_usage.png',
 'remaining_tokens_by_prompt.png',
 'strategy_comparison.png',
 'token_budget_by_section.png',
 'token_count_by_category_and_tokenizer.png',
 'tokenizer_comparison_summary.png',
 'tokens_per_word_by_category_and_tokenizer.png']

## Why the Formula Matters

The key formula is:

`prompt tokens + reserved output tokens <= context window`

A prompt may look small enough by itself, but if you reserve space for the model's answer, the total can grow beyond the limit. That is why output reservation matters.

This especially matters for:
- **RAG prompts**, because retrieved chunks add many input tokens
- **chat history**, because previous turns accumulate over time
- **agents and tool calls**, because tool schemas and structured data add overhead
- **code prompts**, because code and logs often tokenize densely

## What the Graphs Show

1. **Context used percent by prompt**
   This compares how much of the available window each prompt consumes.

2. **Remaining tokens by prompt**
   This shows how much headroom is left after accounting for the prompt and the reserved answer size.

3. **Average context used percent by tokenizer**
   This compares tokenizer efficiency for the same prompts.

4. **Fits vs exceeds context**
   This shows how many prompt setups remain valid and how many go over the limit.

## Step 3 Takeaway

Prompt size alone is not enough. You must leave room for output tokens too. Context windows are shared by input and output, so a prompt that fits by itself may still fail when answer space is reserved.